In [1]:
!pip install sktime

import os
import numpy as np
import pandas as pd
import librosa
import kagglehub
import time
from sklearn.model_selection import train_test_split
from sklearn.linear_model import RidgeClassifierCV
from IPython.display import Audio, display
from sktime.classification.kernel_based import RocketClassifier
from sktime.transformations.panel.rocket import Rocket
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

zsh:1: command not found: pip


/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ==========================================
# 1. CARGA Y FILTRADO DEL DATASET (ESC-50)
# ==========================================
print("Descarga del dataset desde Kaggle...")
dataset_root_path = kagglehub.dataset_download("mmoreaux/environmental-sound-classification-50")

# Imprimo los nombres de los archivos descargados
print(f"Contenido de la carpeta descargada por Kagglehub: {os.listdir(dataset_root_path)}")

# Guardo la carpeta de metadata
metadata_path = os.path.join(dataset_root_path, 'esc50.csv')

if os.path.exists(metadata_path):
    df = pd.read_csv(metadata_path)
    print("Archivo de metadatos cargado correctamente.")
else:
    raise FileNotFoundError(f"No se encontró el archivo esc50.csv en la ruta: {metadata_path}")

# Mis clases
mis_clases = ['alarm', 'door_bell', 'cat', 'crying_baby', 'dog', 'shouting']
df_filtrado = df[df['category'].isin(mis_clases)].copy()

# Muestras con las etiquetas que busco
print(f"Total de muestras encontradas para tus 6 clases en el CSV: {len(df_filtrado)}")


Descarga del dataset desde Kaggle...
Contenido de la carpeta descargada por Kagglehub: ['esc50.csv', 'audio', 'utils.py', 'utils2.py', 'bc_utils.py']
Archivo de metadatos cargado correctamente.
Total de muestras encontradas para tus 6 clases en el CSV: 120


In [4]:
correct_audio_dir = None

# 1. BÚSQUEDA AUTOMÁTICA DE LA CARPETA
# Escaneamos el directorio raíz para encontrar dónde están realmente los .wav
for root, dirs, files in os.walk(dataset_root_path):
    if any(f.endswith('.wav') for f in files):
        correct_audio_dir = root
        break

# Verificamos si la encontró
if correct_audio_dir is None:
    raise FileNotFoundError(f"❌ ¡Alerta! No se encontró ninguna carpeta con archivos .wav dentro de {dataset_root_path}")


X_list = []
y_list = []

# Asumiendo que ya tenés tu DataFrame df_filtrado cargado en memoria
for index, row in df_filtrado.iterrows():
    # Ajustá 'posible_audio_dir' a la ruta real de tu carpeta de audio
    file_path = os.path.join(correct_audio_dir, row['filename'])

    try:
        # 1. Cargar el audio
        y, sr = librosa.load(file_path, sr=22050)

        # 2. Extraer MFCCs
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)

        # 3. Achatar a 2D (Vector plano de 40 números por audio)
        mfccs_scaled_features = np.mean(mfccs.T, axis=0)

        X_list.append(mfccs_scaled_features)
        y_list.append(row['category'])

    except Exception as e:
        print(f"❌ Error procesando {file_path}: {e}")

# Convertir las listas a matrices de Numpy
X = np.array(X_list)
y = np.array(y_list)

# 🛠️ LA SOLUCIÓN MÁGICA: Agregar la dimensión de "canal" para que sktime lo procese
# Pasamos de tener una forma (N, 40) a (N, 1, 40)
X = np.expand_dims(X, axis=1)



In [5]:

# Codificar las etiquetas de texto a números
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Separar en conjuntos de Entrenamiento y Prueba
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

print("🚀 Iniciando transformación con ROCKET...")
inicio = time.time()

# Instanciar y ajustar ROCKET
rocket = Rocket(num_kernels=5000, random_state=42)
rocket.fit(X_train)

# Transformar los audios a través de los kernels aleatorios
X_train_transform = rocket.transform(X_train)
X_test_transform = rocket.transform(X_test)

# Clasificador lineal final
classifier = RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))
classifier.fit(X_train_transform, y_train)

# Evaluar el modelo
score = classifier.score(X_test_transform, y_test)
fin = time.time()

print(f"✅ Precisión (Test Accuracy): {score * 100:.2f}%")
print(f"⏱️ Tiempo total de ejecución: {fin - inicio:.2f} segundos")

🚀 Iniciando transformación con ROCKET...
✅ Precisión (Test Accuracy): 83.33%
⏱️ Tiempo total de ejecución: 0.28 segundos


/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
